# NB01 — What SemCat alone would tell you

**Goal.** Record the SemCat-only answer to Q1 (which BERDL tables annotate gene/protein
function, and over which regions, in which coordinate system, with which identifier
systems), **frozen before ground truth is built in NB02.** The ordering is the
methodological point: grading SemCat against a ground truth derived from SemCat would be
circular, so this notebook must not look at live table schemas at all. It queries
`globalusers.semantic_catalog` and nothing else.

**Discriminating query (run first).** The coordinate-system sub-test. Per `RESEARCH_PLAN.md`
this separates **H1** from **H2+H3**: if SemCat cannot sort its own `coordinate`-labelled
columns into geographic / nucleotide / amino-acid / HMM-model, then H1 ("sufficient
instrument") is dead regardless of any recall number, because coordinate system is
explicitly part of what was asked per table.

In [1]:
import sys, os, re, json
sys.path.insert(0, os.path.abspath("../src"))

import pandas as pd
from semcat import (SEMCAT, ACCESSIBLE_TENANTS, ACCESSIBLE_IN_SEMCAT,
                    RX_FUNC_ID, RX_SPAN, RX_STRAND, RX_SEQREF, RX_GEO,
                    ID_SYSTEMS, classify_coordinate_system)
from berdl_notebook_utils.setup_spark_session import get_spark_session

spark = get_spark_session(app_name="acatlas_nb01")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 48)

os.makedirs("../data", exist_ok=True)
print("SemCat =", SEMCAT)

SemCat = globalusers.semantic_catalog


In [2]:
# Provenance of the catalog snapshot under test.
meta = spark.sql(f"SELECT * FROM {SEMCAT}.catalog_meta").toPandas()
print(meta[["generated_at", "regenerated_at", "n_columns", "n_edges", "code_version"]].to_string(index=False))

scale = spark.sql(f'''
SELECT (SELECT COUNT(*) FROM {SEMCAT}.tables)          AS n_tables,
       (SELECT COUNT(DISTINCT tenant)   FROM {SEMCAT}.tables) AS n_tenants,
       (SELECT COUNT(DISTINCT database) FROM {SEMCAT}.tables) AS n_databases,
       (SELECT COUNT(*) FROM {SEMCAT}.columns)         AS n_columns,
       (SELECT COUNT(*) FROM {SEMCAT}.concepts)        AS n_concepts,
       (SELECT COUNT(*) FROM {SEMCAT}.verified_joins)  AS n_verified_joins
''').toPandas()
print()
print(scale.to_string(index=False))

        generated_at              regenerated_at  n_columns  n_edges code_version
2026-10-06T19:38:26Z 2026-10-07T19:32:16.135360Z      91246    17181       0.27.0



 n_tables  n_tenants  n_databases  n_columns  n_concepts  n_verified_joins
     2509         22          153      91246         119                 6


## 1. Discriminating query — can SemCat resolve coordinate *system*?

Pull every column SemCat labels `content_type='coordinate'`, together with the only
auxiliary fields SemCat offers (`type`, `lower`, `upper`, and the sibling column names in
the same table). Then attempt classification using **those fields only** — no live data,
no outside knowledge of the tables.

`classify_coordinate_system` (in `src/semcat.py`) applies, in order: a name test for
HMM/profile coordinates; a name test for geography; a type+bounds test for geography
(double inside lat/lon range); and for integer spans, a magnitude test (upper bound
> 100,000 cannot be residues) plus a sibling-column test (contig/scaffold → nucleotide,
protein/md5/seq_len → amino acid). Anything it cannot decide is `ambiguous`.

In [3]:
coord = spark.sql(f'''
SELECT tenant, database, table_name, column_name, type, lower, upper,
       facet_layer, facet_confidence, comment, description
FROM {SEMCAT}.columns
WHERE content_type = 'coordinate'
''').toPandas()

# Sibling column names per table — also a SemCat field, so still in-bounds.
sibs = spark.sql(f'''
SELECT tenant, database, table_name, collect_set(column_name) AS cols
FROM {SEMCAT}.columns
GROUP BY tenant, database, table_name
''').toPandas()
sibmap = {(r.tenant, r.database, r.table_name): list(r.cols) for r in sibs.itertuples()}

rows = []
for r in coord.itertuples():
    cols = sibmap.get((r.tenant, r.database, r.table_name), [])
    system, basis = classify_coordinate_system(r.column_name, r.type, r.lower, r.upper, cols)
    rows.append(dict(tenant=r.tenant, database=r.database, table_name=r.table_name,
                     column_name=r.column_name, type=r.type, lower=r.lower, upper=r.upper,
                     semcat_label="coordinate", inferred_system=system, basis=basis,
                     accessible=r.tenant in ACCESSIBLE_TENANTS))
ct = pd.DataFrame(rows)
print(f"columns SemCat labels 'coordinate': {len(ct)}")
print()
print(ct["inferred_system"].value_counts().to_string())

columns SemCat labels 'coordinate': 78

inferred_system
geographic    32
ambiguous     22
nucleotide    12
amino_acid    10
hmm_model      2


In [4]:
# The headline number for the sub-test: how many distinct coordinate SYSTEMS hide
# behind this single label, and how much of the resolution came from the label itself.
n_sys = ct["inferred_system"].nunique()
print(f"distinct coordinate systems sharing the single label 'coordinate': {n_sys}")
print()
print("The `coordinate` label contributes 0 bits toward which system a column is in:")
print("every classification above was decided by type, numeric bounds, or sibling")
print("column names -- never by the label. Basis counts:")
print()
print(ct["basis"].str.replace(r"\d[\d,]*", "N", regex=True).value_counts().to_string())

distinct coordinate systems sharing the single label 'coordinate': 5

The `coordinate` label contributes 0 bits toward which system a column is in:
every classification above was decided by type, numeric bounds, or sibling
column names -- never by the label. Basis counts:

basis
column name is a geographic coordinate                                           32
sibling contig/scaffold column, no protein column                                12
string-typed position with no usable bounds in SemCat                            10
sibling protein/mdN/seq_len column, no contig column                             10
integer span, upper bound unknown, contig_sibling=False protein_sibling=False     6
string-typed position, contig sibling but no bounds in SemCat                     4
integer span, upper bound unknown, contig_sibling=True protein_sibling=True       2
column name says model/hmm                                                        2


In [5]:
# Breakdown by tenant x inferred system, and the geographic/genomic mix.
print(pd.crosstab(ct["tenant"], ct["inferred_system"], margins=True).to_string())
print()
geo = (ct["inferred_system"] == "geographic").sum()
mol = ct["inferred_system"].isin(["nucleotide", "amino_acid", "hmm_model"]).sum()
amb = (ct["inferred_system"] == "ambiguous").sum()
print(f"geographic: {geo}   molecular (nt/aa/model): {mol}   ambiguous: {amb}")
print(f"-> a user filtering content_type='coordinate' to find genome features gets "
      f"{geo}/{len(ct)} = {geo/len(ct):.0%} irrelevant geographic columns.")

inferred_system  ambiguous  amino_acid  geographic  hmm_model  nucleotide  All
tenant                                                                        
arkinlab                 3           0           6          0           0    9
enigma                   2           0           0          0           2    4
globalusers              2           0           0          0           0    2
kbase                    0           4           2          0           0    6
kescience                8           6           0          0          10   24
lambda                   1           0           0          0           0    1
netl                     0           0           2          0           0    2
nmdc                     4           0           4          2           0   10
refdata                  2           0           0          0           0    2
usgs                     0           0          18          0           0   18
All                     22          10          32  

In [6]:
# The ambiguous residue is the part SemCat genuinely cannot answer. Show it,
# because these are the columns NB02 will have to resolve from live data.
amb_df = ct[ct["inferred_system"] == "ambiguous"][
    ["tenant", "database", "table_name", "column_name", "type", "lower", "upper", "basis"]]
print(f"{len(amb_df)} columns unresolvable from SemCat fields alone:")
print(amb_df.to_string(index=False))

22 columns unresolvable from SemCat fields alone:
     tenant                 database                        table_name               column_name   type lower upper                                                                         basis
   arkinlab                    dbcan                         dbcan_pul nucleotide_position_range string   NaN   NaN                         string-typed position with no usable bounds in SemCat
   arkinlab             microbeatlas             enriched_metadata_gee             LatFieldValue string   NaN   NaN                         string-typed position with no usable bounds in SemCat
   arkinlab             microbeatlas             enriched_metadata_gee             LonFieldValue string   NaN   NaN                         string-typed position with no usable bounds in SemCat
     enigma      genome_depot_enigma                      browser_gene                     start    int   NaN   NaN   integer span, upper bound unknown, contig_sibling=True p

In [7]:
ct.to_csv("../data/semcat_coordinate_typing.csv", index=False)
print("wrote ../data/semcat_coordinate_typing.csv", ct.shape)

wrote ../data/semcat_coordinate_typing.csv (78, 11)


### Verdict on the discriminating query

Recorded in the cell below as a plain statement of what the numbers show, so the
pre-registered H1 test is adjudicated here rather than reconstructed later.

In [8]:
h1_coord_pass = (n_sys == 1)
print(f"H1 required: coordinate system resolvable from SemCat fields alone.")
print(f"Observed: {n_sys} distinct systems share the one `coordinate` label; "
      f"{amb} of {len(ct)} columns ({amb/len(ct):.0%}) are unresolvable even after")
print(f"bringing in type, bounds, and sibling columns.")
print()
print(f"H1 coordinate sub-test: {'PASS' if h1_coord_pass else 'FAIL'}")

H1 required: coordinate system resolvable from SemCat fields alone.
Observed: 5 distinct systems share the one `coordinate` label; 22 of 78 columns (28%) are unresolvable even after
bringing in type, bounds, and sibling columns.

H1 coordinate sub-test: FAIL


## 2. SemCat-only candidate discovery

Two routes SemCat offers, recorded separately so NB03 can grade each:

- **Route A — the semantic layer** (`content_type`): what SemCat *means*. This is the
  route the catalog is built to support.
- **Route B — name regex over `semantic_catalog.columns`**: uses SemCat only as a
  column-name inventory, ignoring every semantic label. This is the **H0 comparator** —
  if B ⊇ A with nothing added, the semantic layer earns nothing.

Both routes see the same 2,599 tables, so the comparison isolates the semantic layer
from the coverage advantage.

In [9]:
# How much of the semantic layer is even populated? This caps Route A's recall.
cov = spark.sql(f'''
SELECT COUNT(*) AS total,
       SUM(CASE WHEN content_type IS NULL THEN 1 ELSE 0 END) AS content_type_null,
       SUM(CASE WHEN concept_label IS NULL THEN 1 ELSE 0 END) AS concept_null,
       SUM(CASE WHEN comment IS NULL OR comment = '' THEN 1 ELSE 0 END) AS comment_null
FROM {SEMCAT}.columns
''').toPandas()
t = int(cov["total"][0])
print(cov.to_string(index=False))
print()
for c in ["content_type_null", "concept_null", "comment_null"]:
    print(f"{c:20s} {int(cov[c][0]):>7,} / {t:,} = {int(cov[c][0])/t:.1%}")

 total  content_type_null  concept_null  comment_null
 91246              76239         88239         79284

content_type_null     76,239 / 91,246 = 83.6%
concept_null          88,239 / 91,246 = 96.7%
comment_null          79,284 / 91,246 = 86.9%


In [10]:
# The concepts layer: is there ANY molecular vocabulary in it?
con = spark.sql(f"SELECT cluster_id, concept_label, n_columns FROM {SEMCAT}.concepts").toPandas()
RX_MOLECULAR = (r"pfam|interpro|kegg|^ko|enzyme|^ec|^go|ontolog|cog|domain|motif|operon|"
                r"bgc|gene|protein|function|annotat|coordinate|strand|contig|genom")
hits = con[con["concept_label"].str.lower().str.contains(RX_MOLECULAR, regex=True, na=False)]
print(f"{len(con)} concepts total; {len(hits)} with any molecular/annotation term in the label")
print()
print("all concept labels, by frequency:")
print(con["concept_label"].value_counts().to_string())

119 concepts total; 1 with any molecular/annotation term in the label

all concept labels, by frequency:
concept_label
temperature       15
concentration     14
microplate        13
depth             10
nitrate            6
air                5
salinity           4
water              3
manganese          3
copper             3
conductivity       2
zinc               2
turbidity          2
sulfate            1
gc                 1
organic_carbon     1
nitrite            1
specific           1
partial            1
amount             1
occup              1
efficiency         1
exp                1
freq               1
lbc                1
blood              1
slope              1
has                1
th                 1
iron               1
feii               1
re                 1
ph                 1
aluminum           1
ammonium           1
room               1
ca                 1
na                 1
m                  1
pressure           1
cond               1
as                 1

In [11]:
# ---- Route A: the semantic layer ----
# Which content_type values are relevant to the question, decided from the vocabulary
# SemCat actually uses (enumerated, not guessed).
all_ct = [r[0] for r in spark.sql(
    f"SELECT DISTINCT content_type FROM {SEMCAT}.columns WHERE content_type IS NOT NULL").collect()]

# Split the function-ish vocabulary into SPECIFIC and GENERIC. This matters:
# `identifier` labels columns in 1,691 of 2,599 tables (65%) and `annotation_text`
# in 1,242 (48%), so including them makes "SemCat found an annotation table" nearly
# vacuous. The split is decided here from SemCat's vocabulary alone, before any
# ground truth exists (NB02), so it cannot be tuned to flatter or damage the result.
RX_CT_FUNC_SPECIFIC = (
    r"pfam|interpro|kegg|^ko_|ko_term|^ec_|ec_number|^go_|go_term|go_namespace|cog|"
    r"ontolog|uniprot|metacyc|rhea|bgc|product_prediction|"
    r"protein_family|domain|operon|gene_cluster|functional_group")
RX_CT_FUNC_GENERIC = r"^identifier$|^annotation_text$|^accession$|^name$|^label$|^category$"
RX_CT_FUNC = RX_CT_FUNC_SPECIFIC + "|" + RX_CT_FUNC_GENERIC
RX_CT_SPAN = r"^coordinate$|^bbox_coordinate$|^length$"
RX_CT_SEQ  = r"contig_id|sequence_id|genome_id|protein_id|gene_id|gene_locus_id|locus_tag|organism_id"

ct_func      = sorted(c for c in all_ct if re.search(RX_CT_FUNC, c.lower()))
ct_func_spec = sorted(c for c in all_ct if re.search(RX_CT_FUNC_SPECIFIC, c.lower()))
ct_func_gen  = sorted(c for c in all_ct if re.search(RX_CT_FUNC_GENERIC, c.lower()))
ct_span = sorted(c for c in all_ct if re.search(RX_CT_SPAN, c.lower()))
ct_seq  = sorted(c for c in all_ct if re.search(RX_CT_SEQ, c.lower()))
print(f"{len(all_ct)} distinct content_type values in use")
print(f"\nfunction-ish TOTAL ({len(ct_func)})")
print(f"\n  SPECIFIC ({len(ct_func_spec)}): {ct_func_spec}")
print(f"\n  GENERIC  ({len(ct_func_gen)}): {ct_func_gen}")
print(f"\nspan-ish ({len(ct_span)}): {ct_span}")
print(f"\nseqref-ish ({len(ct_seq)}): {ct_seq}")

464 distinct content_type values in use

function-ish TOTAL (43)

  SPECIFIC (37): ['bgc_class', 'bgc_region_id', 'cog_category', 'cog_id', 'domain_id', 'ec_number', 'functional_group_score', 'gene_cluster_id', 'go_namespace', 'go_term', 'interpro_id', 'kegg_gene_id', 'kegg_id', 'kegg_module', 'kegg_module_id', 'kegg_organism', 'kegg_pathway', 'kegg_pathway_id', 'kegg_rclass', 'kegg_reaction', 'kegg_reaction_id', 'kegg_tc', 'kegg_term', 'ko_term', 'metacyc_protein_id', 'metacyc_reaction_id', 'ontology_term', 'ontology_term_id', 'ontology_term_label', 'operon_id', 'pfam_id', 'pfam_name', 'product_prediction', 'protein_family_id', 'rhea_id', 'uniprot_accession', 'uniprot_id']

  GENERIC  (6): ['accession', 'annotation_text', 'category', 'identifier', 'label', 'name']

span-ish (3): ['bbox_coordinate', 'coordinate', 'length']

seqref-ish (11): ['contig_id', 'gene_id', 'gene_locus_id', 'genome_id', 'kegg_gene_id', 'locus_tag', 'metacyc_protein_id', 'organism_id', 'protein_id', 'sequence_id

In [12]:
def q_list(vals):
    return ", ".join("'" + v.replace("'", "''") + "'" for v in vals)

route_a = spark.sql(f'''
SELECT tenant, database, table_name,
       MAX(CASE WHEN content_type IN ({q_list(ct_func)}) THEN 1 ELSE 0 END) AS a_func,
       MAX(CASE WHEN content_type IN ({q_list(ct_func_spec)}) THEN 1 ELSE 0 END) AS a_func_specific,
       MAX(CASE WHEN content_type IN ({q_list(ct_func_gen)}) THEN 1 ELSE 0 END) AS a_func_generic,
       MAX(CASE WHEN content_type IN ({q_list(ct_span)}) THEN 1 ELSE 0 END) AS a_span,
       MAX(CASE WHEN content_type = 'strand' THEN 1 ELSE 0 END)             AS a_strand,
       MAX(CASE WHEN content_type IN ({q_list(ct_seq)}) THEN 1 ELSE 0 END)  AS a_seqref
FROM {SEMCAT}.columns
GROUP BY tenant, database, table_name
''').toPandas()
route_a["route_a_candidate"] = ((route_a.a_func == 1) | (route_a.a_span == 1)).astype(int)
route_a["route_a_specific"] = ((route_a.a_func_specific == 1) | (route_a.a_span == 1)).astype(int)
print(f"Route A, any label      : {route_a.route_a_candidate.sum()} candidate tables of {len(route_a)} indexed")
print(f"Route A, SPECIFIC labels: {route_a.route_a_specific.sum()} candidate tables")
print()
print(route_a[["a_func", "a_func_specific", "a_func_generic", "a_span", "a_strand", "a_seqref"]].sum().to_string())
print()
print("The gap between the two Route A figures is the generic-label inflation:")
print(f"  {route_a.route_a_candidate.sum() - route_a.route_a_specific.sum()} tables qualify ONLY via")
print("  `identifier`/`annotation_text`/`accession`/`name`/`label`/`category`, which say")
print("  nothing about gene or protein function. NB03 grades the specific set.")

Route A, any label      : 2007 candidate tables of 2435 indexed
Route A, SPECIFIC labels: 656 candidate tables

a_func             1994
a_func_specific     603
a_func_generic     1961
a_span               65
a_strand              5
a_seqref            280

The gap between the two Route A figures is the generic-label inflation:
  1351 tables qualify ONLY via
  `identifier`/`annotation_text`/`accession`/`name`/`label`/`category`, which say
  nothing about gene or protein function. NB03 grades the specific set.


In [13]:
# ---- Route B: name regex over SemCat's column inventory (H0 comparator) ----
route_b = spark.sql(f'''
SELECT tenant, database, table_name,
       MAX(CASE WHEN lower(column_name) RLIKE '{RX_FUNC_ID}' THEN 1 ELSE 0 END) AS b_func,
       MAX(CASE WHEN lower(column_name) RLIKE '{RX_SPAN}'
                 AND NOT lower(column_name) RLIKE '{RX_GEO}' THEN 1 ELSE 0 END) AS b_span,
       MAX(CASE WHEN lower(column_name) RLIKE '{RX_STRAND}' THEN 1 ELSE 0 END)  AS b_strand,
       MAX(CASE WHEN lower(column_name) RLIKE '{RX_SEQREF}' THEN 1 ELSE 0 END)  AS b_seqref
FROM {SEMCAT}.columns
GROUP BY tenant, database, table_name
''').toPandas()
route_b["route_b_candidate"] = ((route_b.b_func == 1) | (route_b.b_span == 1)).astype(int)
print(f"Route B (name regex): {route_b.route_b_candidate.sum()} candidate tables")
print(route_b[["b_func", "b_span", "b_strand", "b_seqref"]].sum().to_string())

Route B (name regex): 758 candidate tables
b_func      719
b_span      102
b_strand     62
b_seqref    723


In [14]:
# Merge the routes with table-level metadata; this frame is the frozen SemCat answer.
tbl = spark.sql(f'''
SELECT tenant, database, table_name, fq, row_count, n_columns, schema_sig,
       comment_coverage, description, description_source, provenance_class, duplicate_of
FROM {SEMCAT}.tables
''').toPandas()

cand = (tbl.merge(route_a, on=["tenant", "database", "table_name"], how="left")
           .merge(route_b, on=["tenant", "database", "table_name"], how="left"))
for c in ["a_func","a_func_specific","a_func_generic","a_span","a_strand","a_seqref",
          "route_a_candidate","route_a_specific",
          "b_func","b_span","b_strand","b_seqref","route_b_candidate"]:
    cand[c] = cand[c].fillna(0).astype(int)
cand["accessible"] = cand["tenant"].isin(ACCESSIBLE_TENANTS)

print(f"{len(cand)} tables in SemCat")
print(pd.crosstab(cand.route_a_candidate, cand.route_b_candidate,
                  rownames=["route_a"], colnames=["route_b"], margins=True).to_string())

2509 tables in SemCat
route_b     0    1   All
route_a                 
0         480   22   502
1        1271  736  2007
All      1751  758  2509


In [15]:
# The H0 test, stated numerically: does the semantic layer add any table the
# name regex missed?
only_a = cand[(cand.route_a_candidate == 1) & (cand.route_b_candidate == 0)]
only_b = cand[(cand.route_a_candidate == 0) & (cand.route_b_candidate == 1)]
only_as = cand[(cand.route_a_specific == 1) & (cand.route_b_candidate == 0)]
print(f"tables found ONLY by the semantic layer, SPECIFIC labels: {len(only_as)}")
print(f"tables found ONLY by the semantic layer (Route A): {len(only_a)}")
print(f"tables found ONLY by the name regex   (Route B): {len(only_b)}")
if len(only_a):
    print()
    print(only_a[["tenant","database","table_name","row_count","a_func","a_span"]]
          .head(30).to_string(index=False))

tables found ONLY by the semantic layer, SPECIFIC labels: 379
tables found ONLY by the semantic layer (Route A): 1271
tables found ONLY by the name regex   (Route B): 22

  tenant            database                            table_name  row_count  a_func  a_span
  enigma               coral                      ddt_brick0001226      27744       1       0
    nmdc     ncbi_biosamples                  env_triads_flattened   20721046       1       0
arkinlab              envdbs                 ecotapestry_lithology     338939       1       0
arkinlab              envdbs         ecotapestry_lithology_0_25deg     338939       1       0
arkinlab   function_junction                       interop_bundles         18       1       0
arkinlab   function_junction                          koros_result          1       1       0
arkinlab   function_junction                 uniref50_carrier_taxa  157125808       1       0
arkinlab          igem_parts                        igem_new_parts          2

In [16]:
# Coverage advantage: what SemCat shows that this account's own inventory cannot.
by_access = cand.groupby("accessible")["route_b_candidate"].agg(["sum", "count"])
by_access.columns = ["candidate_tables", "all_tables"]
print(by_access.to_string())
print()
inacc = cand[(~cand.accessible) & (cand.route_b_candidate == 1)]
print(f"candidate annotation tables in tenants this account CANNOT read: {len(inacc)}")
print(f"across {inacc.tenant.nunique()} tenants, {inacc.database.nunique()} databases")
print()
print(inacc.groupby("tenant").size().sort_values(ascending=False).to_string())

            candidate_tables  all_tables
accessible                              
False                    493        1851
True                     265         658

candidate annotation tables in tenants this account CANNOT read: 493
across 15 tenants, 59 databases

tenant
phagefoundry              230
enigma                     77
protect                    51
kbaseincubator             39
arkinlab                   24
planetmicrobe              15
usgs                       14
plantmicrobeinterfaces     13
lambda                      9
ese                         6
globalusers                 6
netl                        4
enigmay12                   2
msyscolo                    2
aiale                       1


In [17]:
# Freeze the SemCat-only answer. NB03 reads this file; it must not be regenerated
# after ground truth exists.
cand.to_csv("../data/semcat_candidates.csv", index=False)
print("wrote ../data/semcat_candidates.csv", cand.shape)

summary = dict(
    semcat_generated_at=str(meta["generated_at"][0]),
    semcat_code_version=str(meta["code_version"][0]),
    n_tables=int(scale["n_tables"][0]), n_tenants=int(scale["n_tenants"][0]),
    n_databases=int(scale["n_databases"][0]), n_columns=int(scale["n_columns"][0]),
    n_concepts=int(scale["n_concepts"][0]), n_verified_joins=int(scale["n_verified_joins"][0]),
    content_type_null_frac=float(int(cov["content_type_null"][0]) / t),
    n_molecular_concepts=int(len(hits)),
    coord_label_columns=int(len(ct)),
    coord_label_distinct_systems=int(n_sys),
    coord_label_geographic=int(geo), coord_label_molecular=int(mol),
    coord_label_ambiguous=int(amb),
    h1_coordinate_subtest_pass=bool(h1_coord_pass),
    route_a_candidates=int(cand.route_a_candidate.sum()),
    route_a_specific_candidates=int(cand.route_a_specific.sum()),
    generic_label_inflation=int(cand.route_a_candidate.sum() - cand.route_a_specific.sum()),
    route_b_candidates=int(cand.route_b_candidate.sum()),
    only_route_a=int(len(only_a)), only_route_a_specific=int(len(only_as)),
    only_route_b=int(len(only_b)),
    candidates_in_inaccessible_tenants=int(len(inacc)),
)
with open("../data/nb01_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

wrote ../data/semcat_candidates.csv (2509, 26)
{
  "semcat_generated_at": "2026-10-06T19:38:26Z",
  "semcat_code_version": "0.27.0",
  "n_tables": 2509,
  "n_tenants": 22,
  "n_databases": 153,
  "n_columns": 91246,
  "n_concepts": 119,
  "n_verified_joins": 6,
  "content_type_null_frac": 0.8355325164938737,
  "n_molecular_concepts": 1,
  "coord_label_columns": 78,
  "coord_label_distinct_systems": 5,
  "coord_label_geographic": 32,
  "coord_label_molecular": 24,
  "coord_label_ambiguous": 22,
  "h1_coordinate_subtest_pass": false,
  "route_a_candidates": 2007,
  "route_a_specific_candidates": 656,
  "generic_label_inflation": 1351,
  "route_b_candidates": 758,
  "only_route_a": 1271,
  "only_route_a_specific": 379,
  "only_route_b": 22,
  "candidates_in_inaccessible_tenants": 493
}
